# Stage 2: look at real price history

We will download up to 20 years of daily prices for 5 well-known stocks, save each one as its own file, and look at what we got. Run the cells from top to bottom, one at a time.

Make sure the kernel at the top right says **RiskWise (.venv)**.

In [ ]:
# Cell 1: tools, and the five stocks we start with
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path

STOCKS = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "SBIN.NS"]   # .NS means NSE
YEARS = 20
DATA = Path("../data/test")
DATA.mkdir(parents=True, exist_ok=True)
start = pd.Timestamp.today().normalize() - pd.DateOffset(years=YEARS)
print("Asking for history from", start.date())

In [ ]:
# Cell 2: download each stock and save it as its own file (this takes a little while)
tables = {}
for sym in STOCKS:
    df = yf.Ticker(sym).history(start=start, auto_adjust=True)
    df.index = df.index.tz_localize(None)   # drop the time-zone label; we only need the date
    df.index.name = "date"
    df.to_csv(DATA / f"{sym}.csv")
    tables[sym] = df
    print(f"{sym:12s} {len(df):5d} trading days, from {df.index[0].date()} to {df.index[-1].date()}")

A **trading day** is a day the market was open. That is why 20 years gives about 4,900 rows, not 7,300: weekends and holidays have no row.

Now let us look inside one table.

In [ ]:
# Cell 3: the first and last few rows for Reliance
reliance = tables["RELIANCE.NS"]
display(reliance.head(3))
display(reliance.tail(3))

Each row is one trading day:
- **Open / Close**: price when the market opened and closed
- **High / Low**: the highest and lowest price that day
- **Volume**: how many shares changed hands
- **Dividends**: money the company paid out per share that day (usually 0)
- **Stock Splits**: 0 on normal days; a number like 2 when each share was split into two (or a bonus issue happened)

In [ ]:
# Cell 4: Reliance's closing price over the whole period
reliance["Close"].plot(figsize=(10, 4), title="Reliance: closing price (adjusted)")
plt.ylabel("price (Rs)")
plt.show()

**Why the early prices look so low:** these are *adjusted* prices. When a company splits its shares, old prices are scaled down so the history stays fair, and that is why Reliance looks cheap 20 years ago. The next cell shows the days when this happened, and the prices around the first one. Look for a sudden 50% drop. There should not be one.

In [ ]:
# Cell 5: days when a split or bonus issue happened, and the prices around the first one
splits = reliance[reliance["Stock Splits"] != 0]
display(splits[["Close", "Stock Splits"]])
if len(splits):
    d = splits.index[0]
    display(reliance.loc[d - pd.Timedelta(days=5): d + pd.Timedelta(days=5), ["Close", "Stock Splits"]])

In [ ]:
# Cell 6: daily return = how much the price changed since yesterday, in percent
# (the building block of volatility)
returns = reliance["Close"].pct_change() * 100
print(returns.describe().round(2))
print("\nFive biggest one-day drops:")
print(returns.nsmallest(5).round(2))

In [ ]:
# Cell 7: quick health check of all five stocks
rows = []
for sym, df in tables.items():
    rows.append({
        "stock": sym,
        "trading days": len(df),
        "first day": df.index[0].date(),
        "zero-volume days": int((df["Volume"] == 0).sum()),
        "missing prices": int(df["Close"].isna().sum()),
    })
pd.DataFrame(rows)

## Your turn

1. In Cell 1, add a more unusual stock to the list, for example `"SUZLON.NS"` (a jumpier one), and re-run Cells 1 to 7 (change `reliance` to that stock in Cells 3 to 6 if you want to inspect it).
2. Look at Cell 6. What is the biggest one-day drop, and does that look like a stock that is calm or jumpy?
3. In Cell 7, does every stock have 20 years of data? Which one has fewer days, and why might that be?